In [ ]:
# !pip install --upgrade pip
# !pip install pandas numpy scipy matplotlib seaborn

## Importações de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')

# Semente aleatória — garante que os resultados sejam
# idênticos toda vez que o notebook for executado
np.random.seed(42)

---
## Módulo 01

### O que é um Teste de Hipótese?

Um **teste de hipótese** é um procedimento estatístico que permite tomar decisões baseadas em dados, controlando a probabilidade de cometer erros.

A lógica é sempre a mesma:

1. Formulamos duas hipóteses opostas sobre a população
2. Coletamos uma amostra
3. Calculamos uma estatística de teste
4. Decidimos se os dados são suficientemente improváveis sob H₀

---

### As duas hipóteses

| Hipótese | Nome | Significado |
|---|---|---|
| **H₀** | Hipótese nula | Status quo — o que assumimos verdadeiro até prova em contrário |
| **H₁** | Hipótese alternativa | O que queremos demonstrar — a afirmação do pesquisador |

> A lógica é similar ao sistema jurídico: o réu é **inocente até prova em contrário**. H₀ é a inocência — só a rejeitamos se houver evidência forte o suficiente.

---

### O p-valor

O **p-valor** responde à pergunta:

> *"Se H₀ fosse verdadeira, qual seria a probabilidade de observar um resultado tão extremo quanto o que obtive?"*

- **p pequeno (< α)** → resultado improvável sob H₀ → **rejeita H₀**
- **p grande (≥ α)** → resultado plausível sob H₀ → **não rejeita H₀**

O nível de significância **α = 0,05** é o limiar mais usado: toleramos 5% de chance de rejeitar H₀ quando ela é verdadeira.

---

### Tipos de erro

| | H₀ verdadeira | H₀ falsa |
|---|---|---|
| **Rejeita H₀** | ❌ Erro Tipo I (falso positivo) — probabilidade = α | ✅ Decisão correta |
| **Não rejeita H₀** | ✅ Decisão correta | ❌ Erro Tipo II (falso negativo) — probabilidade = β |

---

### Testes unicaudal vs. bicaudal

| Tipo | H₁ | Quando usar |
|---|---|---|
| **Bicaudal** | μ₁ ≠ μ₂ | Qualquer diferença, em qualquer direção |
| **Unicaudal direita** | μ₁ > μ₂ | Esperamos que o grupo 1 seja **maior** |
| **Unicaudal esquerda** | μ₁ < μ₂ | Esperamos que o grupo 1 seja **menor** |

---
## Módulo 02

### Caso 1 — Teste t para Amostras Independentes
#### A campanha de marketing aumentou o ticket médio?

**Contexto:** uma empresa rodou uma campanha de marketing e quer saber se o ticket médio dos clientes **depois** da campanha é maior do que **antes**.

Como os grupos são **independentes** (clientes distintos antes e depois), usamos o **Teste t de Student para amostras independentes** — `ttest_ind()`.

**Hipóteses:**

| | |
|---|---|
| **H₀** | μ_antes ≥ μ_depois — a campanha não aumentou o ticket médio |
| **H₁** | μ_antes < μ_depois — a campanha **aumentou** o ticket médio |

> Teste **unicaudal à esquerda** — verificamos se `antes` é menor que `depois`.
>
> Por isso usamos `alternative='less'` no scipy: estamos testando se o **primeiro argumento** (antes) é menor que o segundo (depois).

In [ ]:
# Simulação de dados de ticket médio
# np.random.normal(loc=média, scale=desvio_padrão, size=n_amostras)
ticket_antes  = np.random.normal(loc=250, scale=60, size=200)
ticket_depois = np.random.normal(loc=265, scale=62, size=200)

print(f'Ticket médio ANTES:  R$ {ticket_antes.mean():.2f}  (DP: {ticket_antes.std():.2f})')
print(f'Ticket médio DEPOIS: R$ {ticket_depois.mean():.2f}  (DP: {ticket_depois.std():.2f})')
print(f'Diferença observada: R$ {ticket_depois.mean() - ticket_antes.mean():.2f}')

#### Visualização das distribuições

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogramas sobrepostos
axes[0].hist(ticket_antes,  bins=25, alpha=0.6, color='steelblue', label='Antes')
axes[0].hist(ticket_depois, bins=25, alpha=0.6, color='darkorange', label='Depois')
axes[0].axvline(ticket_antes.mean(),  color='steelblue',  linestyle='--', linewidth=1.5)
axes[0].axvline(ticket_depois.mean(), color='darkorange', linestyle='--', linewidth=1.5)
axes[0].set_title('Distribuição do Ticket Médio')
axes[0].set_xlabel('Ticket (R$)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

# Boxplot comparativo
axes[1].boxplot([ticket_antes, ticket_depois],
                labels=['Antes', 'Depois'],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Boxplot — Antes vs Depois')
axes[1].set_ylabel('Ticket (R$)')

plt.tight_layout()
plt.show()

#### Aplicar o teste

In [ ]:
# ttest_ind() — amostras independentes
# alternative='less' → H1: ticket_antes < ticket_depois
# (o primeiro argumento é o grupo que testamos ser menor)
t_stat, p_valor = stats.ttest_ind(ticket_antes, ticket_depois, alternative='less')

print(f't-statistic : {t_stat:.4f}')
print(f'p-valor     : {p_valor:.4f}')

In [ ]:
alpha = 0.05

print('=' * 50)
print('CASO 1 — Campanha de Marketing')
print('=' * 50)
print(f'H₀: μ_antes ≥ μ_depois')
print(f'H₁: μ_antes < μ_depois  (unicaudal esquerda)')
print(f'α  = {alpha}')
print(f'p  = {p_valor:.4f}')
print()
if p_valor < alpha:
    print('✅ Rejeita H₀')
    print('   Há evidência estatística de que a campanha aumentou o ticket médio.')
else:
    print('❌ Não rejeita H₀')
    print('   Sem evidência suficiente de aumento no ticket médio.')